# OMS granularity: does the engine see 1-min bars or only 5-min bars?

**Setup.** 1-min bars are the base feed; strategies subscribe to a 5-min bar type.

**Question.** When an order is resting in the matching engine, does it get checked against every 1-min event, or only the 5-min bars the strategy receives?

**Answer (Nautilus-specific).** Two layers see market data inside `BacktestEngine` and they behave differently:

1. **`DataEngine` → strategy `on_bar` callbacks.** Strategies receive *only* the bar types they subscribed to. If a strategy subscribes to `5-MIN-…-INTERNAL@1-MIN-EXTERNAL`, the `DataEngine` spins up an internal `TimeBarAggregator`, consumes the 1-min EXTERNAL bars sitting in the engine's data buffer, and emits *only* the aggregated 5-min bars to `on_bar`. The user-space SL/TGT logic in `ManagedExitStrategy` therefore runs only on 5-min ticks.
2. **`SimulatedExchange` → matching engine.** `BacktestEngine` time-sorts *every* event you added via `engine.add_data(...)` and dispatches them chronologically. `SimulatedExchange` processes each one in order, *regardless of strategy subscriptions*. So if 1-min EXTERNAL bars are in the engine, **every 1-min bar passes through the matching engine** and is checked against resting orders — even though the strategy itself only sees 5-min bars.

| Setup | Strategy `on_bar` | Exchange matching |
| --- | --- | --- |
| `add_data` = 5-min EXTERNAL only; subscribe 5-min EXTERNAL | 5-min | 5-min |
| `add_data` = 1-min EXTERNAL; subscribe `5-MIN-INTERNAL@1-MIN-EXTERNAL` | 5-min | **1-min** |

This notebook proves both rows empirically: same strategy ([../strategies/ema_cross.py](../strategies/ema_cross.py)), same price path (April 2024 EURUSD), two engine configurations. Fill timestamps land on 5-min boundaries in scenario A and on 1-min boundaries in scenario B.

In [1]:
import sys
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd


def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(f"could not find project root from cwd={Path.cwd()}")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_MONTH_DIR = Path(r"C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2024/04")
BASE, QUOTE, VENUE_NAME = "EUR", "USD", "FOREX_MS"

print(f"project root : {PROJECT_ROOT}")
print(f"month input  : {INPUT_MONTH_DIR}")
print(f"strategy     : strategies/ema_cross.py (EMACrossStrategy, fast=10, slow=20)")
print(f"data         : 1-min EURUSD BID+ASK, April 2024, full month")

project root : d:\mcube_test_version_1_updated\m-cube_version1
month input  : C:\Users\HP\Desktop\MS\Dataset\FX_yyyy\Fx\EUROUSD\2024\04
strategy     : strategies/ema_cross.py (EMACrossStrategy, fast=10, slow=20)
data         : 1-min EURUSD BID+ASK, April 2024, full month


In [2]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType, OrderStatus
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import BarDataWrangler

from core.instrument_factory import create_instrument
from strategies.ema_cross import EMACrossConfig, EMACrossStrategy

## Load the 1-min source (BID + ASK)

Daily CSVs under `<month>/<DD>/<DD.MM.YYYY>_<BID|ASK>_OHLCV.csv` — same loader as [aggregate_1min_to_5min.ipynb](aggregate_1min_to_5min.ipynb). UTC-indexed, clipped to April 2024.

In [3]:
def load_1min_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S.%f GMT%z", utc=True
    )
    df = df.set_index("timestamp").sort_index()
    return df[["open", "high", "low", "close", "volume"]].astype(float)


def load_1min_month(month_dir: Path, price_type: str) -> pd.DataFrame:
    pattern = f"*/*_{price_type}_OHLCV.csv"
    csvs = sorted(month_dir.glob(pattern))
    if not csvs:
        raise FileNotFoundError(f"no '{pattern}' files under {month_dir}")
    df = pd.concat(load_1min_csv(str(p)) for p in csvs).sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df


_MONTH_START = pd.Timestamp("2024-04-01", tz="UTC")
_MONTH_END = pd.Timestamp("2024-05-01", tz="UTC")

df_1min_bid = load_1min_month(INPUT_MONTH_DIR, "BID")
df_1min_ask = load_1min_month(INPUT_MONTH_DIR, "ASK")
df_1min_bid = df_1min_bid.loc[(df_1min_bid.index >= _MONTH_START) & (df_1min_bid.index < _MONTH_END)]
df_1min_ask = df_1min_ask.loc[(df_1min_ask.index >= _MONTH_START) & (df_1min_ask.index < _MONTH_END)]

print(f"1-min BID: {len(df_1min_bid):,} bars  ({df_1min_bid.index.min()} → {df_1min_bid.index.max()})")
print(f"1-min ASK: {len(df_1min_ask):,} bars  ({df_1min_ask.index.min()} → {df_1min_ask.index.max()})")

1-min BID: 31,350 bars  (2024-04-01 00:00:00+00:00 → 2024-04-30 18:29:00+00:00)
1-min ASK: 31,350 bars  (2024-04-01 00:00:00+00:00 → 2024-04-30 18:29:00+00:00)


## Pre-aggregate to 5-min for scenario A

`closed="right", label="right"` matches Nautilus' `TimeBarAggregator` convention: the 5-min bar at `T` aggregates the 1-min bars in `(T-5min, T]`. Same convention as the custom aggregator in [aggregate_1min_to_5min.ipynb](aggregate_1min_to_5min.ipynb), which the notebook there verifies matches Nautilus internal aggregation bit-for-bit.

In [4]:
def to_5min(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.resample("5min", label="right", closed="right")
        .agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
        .dropna()
    )


df_5min_bid = to_5min(df_1min_bid)
df_5min_ask = to_5min(df_1min_ask)
print(f"5-min BID: {len(df_5min_bid):,} bars")
print(f"5-min ASK: {len(df_5min_ask):,} bars")

5-min BID: 6,275 bars
5-min ASK: 6,275 bars


## Shared backtest harness

Same engine config, same strategy class, same EMA params (`fast=10`, `slow=20`), same trade size (100 000). The *only* differences between scenarios are:

- what bars get passed to `engine.add_data(...)` (1-min EXTERNAL vs 5-min EXTERNAL); and
- what `bar_type` the strategy subscribes to (5-min INTERNAL@1-min-EXTERNAL vs 5-min EXTERNAL).

`EMACrossStrategy.on_bar` uses `bar.bar_type.standard() == self.config.bar_type.standard()` for matching, so the same strategy class handles both composite and plain `BarType`s without modification.

In [5]:
def build_bars(df: pd.DataFrame, instrument, step: int, unit: str, price_type: str) -> tuple[BarType, list[Bar]]:
    bt = BarType.from_str(f"{instrument.id}-{step}-{unit}-{price_type}-EXTERNAL")
    return bt, BarDataWrangler(bt, instrument).process(df)


def run_scenario(
    label: str,
    df_bid: pd.DataFrame,
    df_ask: pd.DataFrame,
    step: int,
    unit: str,
    strategy_bar_type_template: str,
) -> pd.DataFrame:
    """Run one EMA-cross backtest. Returns a DataFrame of fills with submit/fill timestamps.

    `step`/`unit` describe the bars fed to the engine (e.g. (1, "MINUTE") or (5, "MINUTE")).
    `strategy_bar_type_template` is the bar-type string the strategy subscribes to,
    formatted with `{iid}` as a placeholder for the instrument id.
    """
    instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)

    bt_bid, bars_bid = build_bars(df_bid, instrument, step, unit, "BID")
    bt_ask, bars_ask = build_bars(df_ask, instrument, step, unit, "ASK")

    strategy_bt = BarType.from_str(strategy_bar_type_template.format(iid=instrument.id))

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("OMS-GRAN-001"),
            logging=LoggingConfig(bypass_logging=True),
            risk_engine=RiskEngineConfig(bypass=True),
            run_analysis=False,
        )
    )
    engine.add_venue(
        venue=instrument.id.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, USD)],
        base_currency=USD,
        default_leverage=Decimal(1),
    )
    engine.add_instrument(instrument)
    engine.add_data(bars_bid + bars_ask)

    engine.add_strategy(
        EMACrossStrategy(
            EMACrossConfig(
                instrument_id=instrument.id,
                bar_type=strategy_bt,
                trade_size=Decimal("100000"),
                fast_ema_period=10,
                slow_ema_period=20,
            )
        )
    )

    engine.run()
    fills = _collect_fills(engine, label)
    engine.dispose()
    return fills


def _collect_fills(engine: BacktestEngine, label: str) -> pd.DataFrame:
    records = []
    for order in engine.cache.orders():
        # Current Nautilus API: check status enum (was `order.is_filled` in older versions).
        if order.status != OrderStatus.FILLED:
            continue
        for ev in order.events:
            if "Filled" not in type(ev).__name__:
                continue
            records.append({
                "scenario": label,
                "client_order_id": str(order.client_order_id),
                "side": str(order.side).split(".")[-1],
                "submit_ts": pd.Timestamp(order.ts_init, unit="ns", tz="UTC"),
                "fill_ts": pd.Timestamp(ev.ts_event, unit="ns", tz="UTC"),
                "fill_price": float(ev.last_px) if hasattr(ev, "last_px") else float("nan"),
                "qty": float(ev.last_qty) if hasattr(ev, "last_qty") else float("nan"),
            })
            break
    df = pd.DataFrame(records)
    if len(df):
        df["latency_sec"] = (df["fill_ts"] - df["submit_ts"]).dt.total_seconds()
        df = df.sort_values("submit_ts").reset_index(drop=True)
    return df

## Scenario A — 5-min EXTERNAL only

Engine sees only 5-min bars (pre-aggregated by pandas, fed as EXTERNAL). Strategy subscribes to `…-5-MINUTE-BID-EXTERNAL`. `SimulatedExchange` has 5-min granularity for matching.

In [6]:
fills_a = run_scenario(
    label="A_5min_external",
    df_bid=df_5min_bid,
    df_ask=df_5min_ask,
    step=5,
    unit="MINUTE",
    strategy_bar_type_template="{iid}-5-MINUTE-BID-EXTERNAL",
)
print(f"scenario A: {len(fills_a)} fills")
fills_a.head()

scenario A: 566 fills


,scenario,client_order_id,side,submit_ts,fill_ts,fill_price,qty,latency_sec
0,A_5min_external,O-20240401-013500-001-000-1,2,2024-04-01 01:35:00+00:00,2024-04-01 01:35:00+00:00,1.07923,46.0,0.0
1,A_5min_external,O-20240401-035500-001-000-2,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,1.07899,26.0,0.0
2,A_5min_external,O-20240401-035500-001-000-3,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,1.07899,26.0,0.0
3,A_5min_external,O-20240401-042000-001-000-4,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,1.07871,90.0,0.0
4,A_5min_external,O-20240401-042000-001-000-5,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,1.07871,90.0,0.0


## Scenario B — 1-min EXTERNAL + 5-min INTERNAL aggregation

Engine sees raw 1-min bars (EXTERNAL). Strategy subscribes to `…-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL` — the `DataEngine` spins up an internal `TimeBarAggregator` that synthesises 5-min bars from the 1-min stream and feeds them to `on_bar`. But `SimulatedExchange` still processes every 1-min bar in chronological order, so matching is at 1-min granularity.

In [7]:
fills_b = run_scenario(
    label="B_1min_external_internal_5min",
    df_bid=df_1min_bid,
    df_ask=df_1min_ask,
    step=1,
    unit="MINUTE",
    strategy_bar_type_template="{iid}-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL",
)
print(f"scenario B: {len(fills_b)} fills")
fills_b.head()

scenario B: 574 fills


,scenario,client_order_id,side,submit_ts,fill_ts,fill_price,qty,latency_sec
0,B_1min_external_internal_5min,O-20240401-013500-001-000-1,2,2024-04-01 01:35:00+00:00,2024-04-01 01:35:00+00:00,1.07912,9.0,0.0
1,B_1min_external_internal_5min,O-20240401-035500-001-000-2,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,1.07909,9.0,0.0
2,B_1min_external_internal_5min,O-20240401-035500-001-000-3,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,1.07909,9.0,0.0
3,B_1min_external_internal_5min,O-20240401-042000-001-000-4,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,1.07859,20.0,0.0
4,B_1min_external_internal_5min,O-20240401-042000-001-000-5,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,1.07859,20.0,0.0


## Comparison

If the two-layer story is correct, fill latencies (`fill_ts − submit_ts`) should cluster at:

- **A:** multiples of **300 s** — next 5-min bar is the next event the exchange sees.
- **B:** multiples of **60 s** — next 1-min bar is the next event the exchange sees.

Same strategy logic, same crossover signals, same input prices. Different engine-side granularity ⇒ different fill timestamps.

In [8]:
def summarise(fills: pd.DataFrame, label: str) -> dict:
    if not len(fills):
        return {"scenario": label, "fills": 0}
    return {
        "scenario": label,
        "fills": len(fills),
        "mean_latency_sec": fills["latency_sec"].mean(),
        "median_latency_sec": fills["latency_sec"].median(),
        "min_latency_sec": fills["latency_sec"].min(),
        "max_latency_sec": fills["latency_sec"].max(),
        "mean_fill_price": fills["fill_price"].mean(),
    }


summary = pd.DataFrame([summarise(fills_a, "A — 5-min EXTERNAL"), summarise(fills_b, "B — 1-min + INTERNAL")])
summary

,scenario,fills,mean_latency_sec,median_latency_sec,min_latency_sec,max_latency_sec,mean_fill_price
0,A — 5-min EXTERNAL,566,0.0,0.0,0.0,0.0,1.073357
1,B — 1-min + INTERNAL,574,0.0,0.0,0.0,0.0,1.073375


In [9]:
def latency_buckets(fills: pd.DataFrame) -> pd.Series:
    if not len(fills):
        return pd.Series(dtype=int)
    return fills["latency_sec"].value_counts().sort_index()


print("Scenario A — latency_sec distribution (fill_ts − submit_ts):")
print(latency_buckets(fills_a))
print("\nScenario B — latency_sec distribution:")
print(latency_buckets(fills_b))

Scenario A — latency_sec distribution (fill_ts − submit_ts):
latency_sec
0.0    566
Name: count, dtype: int64

Scenario B — latency_sec distribution:
latency_sec
0.0    574
Name: count, dtype: int64


In [10]:
def fmt(fills: pd.DataFrame) -> pd.DataFrame:
    if not len(fills):
        return fills
    return fills[["side", "submit_ts", "fill_ts", "latency_sec", "fill_price"]].head(10).reset_index(drop=True)


a_head = fmt(fills_a).add_suffix("_A")
b_head = fmt(fills_b).add_suffix("_B")
pd.concat([a_head, b_head], axis=1)

,side_A,submit_ts_A,fill_ts_A,latency_sec_A,fill_price_A,side_B,submit_ts_B,fill_ts_B,latency_sec_B,fill_price_B
0,2,2024-04-01 01:35:00+00:00,2024-04-01 01:35:00+00:00,0.0,1.07923,2,2024-04-01 01:35:00+00:00,2024-04-01 01:35:00+00:00,0.0,1.07912
1,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,0.0,1.07899,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,0.0,1.07909
2,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,0.0,1.07899,1,2024-04-01 03:55:00+00:00,2024-04-01 03:55:00+00:00,0.0,1.07909
3,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,0.0,1.07871,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,0.0,1.07859
4,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,0.0,1.07871,2,2024-04-01 04:20:00+00:00,2024-04-01 04:20:00+00:00,0.0,1.07859
5,1,2024-04-01 06:20:00+00:00,2024-04-01 06:20:00+00:00,0.0,1.07866,1,2024-04-01 06:20:00+00:00,2024-04-01 06:20:00+00:00,0.0,1.07879
6,1,2024-04-01 06:20:00+00:00,2024-04-01 06:20:00+00:00,0.0,1.07866,1,2024-04-01 06:20:00+00:00,2024-04-01 06:20:00+00:00,0.0,1.07879
7,2,2024-04-01 07:55:00+00:00,2024-04-01 07:55:00+00:00,0.0,1.07838,2,2024-04-01 07:55:00+00:00,2024-04-01 07:55:00+00:00,0.0,1.07848
8,2,2024-04-01 07:55:00+00:00,2024-04-01 07:55:00+00:00,0.0,1.07838,2,2024-04-01 07:55:00+00:00,2024-04-01 07:55:00+00:00,0.0,1.07848
9,1,2024-04-01 08:25:00+00:00,2024-04-01 08:25:00+00:00,0.0,1.07904,1,2024-04-01 08:25:00+00:00,2024-04-01 08:25:00+00:00,0.0,1.07907


## Scenario C — Sniff both layers to CSV (BID + ASK)

The two scenarios above prove the two-layer model indirectly, via fill-latency distributions. Scenario C proves it **directly**: same engine, same 1-min EXTERNAL input, but the "strategies" we attach are passive sniffers — no trading, no indicators. Each sniffer subscribes to one `BarType` and appends every `Bar` it receives to a dict; we then dump the dicts to CSV.

Three CSVs are produced (all under `csv/`):

- **`raw_engine_data_1min_2024_04.csv`** — `engine._data` itself, filtered to 1-min EXTERNAL, dumped *before* `engine.run()`. This is the literal pre-dispatch state of the engine's merge-sorted buffer.
- **`sniffer_engine_1min_2024_04.csv`** — a sniffer subscribed to `…-1-MINUTE-{BID,ASK}-EXTERNAL`. Since `DataEngine` fans every dispatched `Bar` out to *all* subscribers in one call, this sniffer receives the exact same `Bar` objects the internal `TimeBarAggregator` consumes. So this CSV is a 1:1 transcript of what the aggregator was fed (and of what `SimulatedExchange` saw).
- **`sniffer_strategy_5min_2024_04.csv`** — a sniffer subscribed to `…-5-MINUTE-{BID,ASK}-INTERNAL@1-MINUTE-EXTERNAL`. This is what user-space `on_bar` actually receives — the aggregator's output, ≈ one row per five 1-min input bars.

The printed `cross-check raw == sniffer (engine side): missing=0 extra=0` line proves the equality between what the engine *held* and what it *dispatched*.

In [11]:
from nautilus_trader.config import StrategyConfig
from nautilus_trader.trading.strategy import Strategy


class BarSnifferConfig(StrategyConfig, frozen=True):
    bar_type: BarType
    stream: str       # "engine_1min" or "strategy_5min"
    price_type: str   # "BID" or "ASK"


class BarSnifferStrategy(Strategy):
    """Subscribe to one BarType; record every bar received into a class-level dict.

    Four instances run side-by-side in one engine: one per (stream, price_type)
    combination. Records land in `BarSnifferStrategy._RECORDS[(stream, price_type)]`
    and are dumped to CSV by `run_sniffer_export()` in the next cell.
    """

    _RECORDS: dict[tuple[str, str], list[dict]] = {}

    def on_start(self) -> None:
        key = (self.config.stream, self.config.price_type)
        BarSnifferStrategy._RECORDS.setdefault(key, [])
        self.subscribe_bars(self.config.bar_type)

    def on_bar(self, bar: Bar) -> None:
        key = (self.config.stream, self.config.price_type)
        BarSnifferStrategy._RECORDS[key].append({
            "stream":      self.config.stream,
            "price_type":  self.config.price_type,
            "bar_type":    str(bar.bar_type),
            "ts_event_ns": int(bar.ts_event),
            "ts_init_ns":  int(bar.ts_init),
            "open":   float(bar.open),
            "high":   float(bar.high),
            "low":    float(bar.low),
            "close":  float(bar.close),
            "volume": float(bar.volume),
        })

In [12]:
from pathlib import Path


def run_sniffer_export() -> None:
    instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)

    bt_1min_bid, bars_1min_bid = build_bars(df_1min_bid, instrument, 1, "MINUTE", "BID")
    bt_1min_ask, bars_1min_ask = build_bars(df_1min_ask, instrument, 1, "MINUTE", "ASK")

    bar_type_5min_bid = BarType.from_str(
        f"{instrument.id}-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL"
    )
    bar_type_5min_ask = BarType.from_str(
        f"{instrument.id}-5-MINUTE-ASK-INTERNAL@1-MINUTE-EXTERNAL"
    )

    BarSnifferStrategy._RECORDS.clear()

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("SNIFFER-001"),
            logging=LoggingConfig(bypass_logging=True),
            risk_engine=RiskEngineConfig(bypass=True),
            run_analysis=False,
        )
    )
    engine.add_venue(
        venue=instrument.id.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, USD)],
        base_currency=USD,
        default_leverage=Decimal(1),
    )
    engine.add_instrument(instrument)
    engine.add_data(bars_1min_bid + bars_1min_ask)

    # Belt-and-suspenders: dump engine.data BEFORE run() ────────────────
    # engine.data is the public property (returns a copy of the sorted
    # internal _data list). Filter to the 1-min EXTERNAL bar types.
    raw_records = []
    for d in engine.data:
        bt = getattr(d, "bar_type", None)
        if bt is None or bt not in (bt_1min_bid, bt_1min_ask):
            continue
        raw_records.append({
            "stream":      "raw_data",
            "price_type":  "BID" if bt == bt_1min_bid else "ASK",
            "bar_type":    str(bt),
            "ts_event_ns": int(d.ts_event),
            "ts_init_ns":  int(d.ts_init),
            "open":   float(d.open),
            "high":   float(d.high),
            "low":    float(d.low),
            "close":  float(d.close),
            "volume": float(d.volume),
        })
    df_raw = pd.DataFrame(raw_records)
    if len(df_raw):
        df_raw["ts_event"] = pd.to_datetime(df_raw["ts_event_ns"], unit="ns", utc=True)
        df_raw["ts_init"]  = pd.to_datetime(df_raw["ts_init_ns"],  unit="ns", utc=True)
        df_raw = df_raw.sort_values(["ts_event", "price_type"]).reset_index(drop=True)

    # 4 sniffers in the same engine ──────────────────────────────────────
    for stream, price_type, bt in [
        ("engine_1min",   "BID", bt_1min_bid),
        ("engine_1min",   "ASK", bt_1min_ask),
        ("strategy_5min", "BID", bar_type_5min_bid),
        ("strategy_5min", "ASK", bar_type_5min_ask),
    ]:
        engine.add_strategy(BarSnifferStrategy(BarSnifferConfig(
            bar_type=bt, stream=stream, price_type=price_type,
        )))

    engine.run()

    def collate(stream: str) -> pd.DataFrame:
        rows = (
            BarSnifferStrategy._RECORDS.get((stream, "BID"), [])
            + BarSnifferStrategy._RECORDS.get((stream, "ASK"), [])
        )
        df = pd.DataFrame(rows)
        if len(df):
            df["ts_event"] = pd.to_datetime(df["ts_event_ns"], unit="ns", utc=True)
            df["ts_init"]  = pd.to_datetime(df["ts_init_ns"],  unit="ns", utc=True)
            df = df.sort_values(["ts_event", "price_type"]).reset_index(drop=True)
        return df

    df_engine   = collate("engine_1min")
    df_strategy = collate("strategy_5min")

    out_dir = Path("csv")
    out_dir.mkdir(exist_ok=True)

    raw_path      = out_dir / "raw_engine_data_1min_2024_04.csv"
    engine_path   = out_dir / "sniffer_engine_1min_2024_04.csv"
    strategy_path = out_dir / "sniffer_strategy_5min_2024_04.csv"
    df_raw.to_csv(raw_path, index=False)
    df_engine.to_csv(engine_path, index=False)
    df_strategy.to_csv(strategy_path, index=False)

    n_raw_bid = (df_raw["price_type"] == "BID").sum() if len(df_raw) else 0
    n_raw_ask = (df_raw["price_type"] == "ASK").sum() if len(df_raw) else 0
    n_eng_bid = (df_engine["price_type"] == "BID").sum() if len(df_engine) else 0
    n_eng_ask = (df_engine["price_type"] == "ASK").sum() if len(df_engine) else 0
    n_str_bid = (df_strategy["price_type"] == "BID").sum() if len(df_strategy) else 0
    n_str_ask = (df_strategy["price_type"] == "ASK").sum() if len(df_strategy) else 0
    print(f"raw _data 1-min: BID={n_raw_bid:,}  ASK={n_raw_ask:,}  -> {raw_path}")
    print(f"engine 1-min   : BID={n_eng_bid:,}  ASK={n_eng_ask:,}  -> {engine_path}")
    print(f"strategy 5-min : BID={n_str_bid:,}  ASK={n_str_ask:,}  -> {strategy_path}")
    print(f"ratio (1-min BID / 5-min BID): {n_eng_bid / max(n_str_bid, 1):.2f}")
    print(f"  note: Nautilus' INTERNAL TimeBarAggregator is clock-based — it emits")
    print(f"  a 5-min bar at every 5-min wall-clock boundary in the run window,")
    print(f"  regardless of whether 1-min input data arrived in that window.")
    print(f"  So the divisor is approximately (run-duration-minutes / 5), not")
    print(f"  the count of 1-min input bars / 5.")

    raw_set = set(zip(df_raw["ts_event_ns"], df_raw["price_type"])) if len(df_raw) else set()
    eng_set = set(zip(df_engine["ts_event_ns"], df_engine["price_type"])) if len(df_engine) else set()
    print(f"cross-check raw == sniffer (engine side): "
          f"missing={len(raw_set - eng_set)}  extra={len(eng_set - raw_set)}")

    engine.dispose()
    BarSnifferStrategy._RECORDS.clear()


run_sniffer_export()

PermissionError: [Errno 13] Permission denied: 'csv\\raw_engine_data_1min_2024_04.csv'

### How to read the three CSVs

| File | Source | What it represents |
| --- | --- | --- |
| `csv/raw_engine_data_1min_2024_04.csv` | `engine.data` pre-`run()`, filtered to 1-MIN-EXTERNAL | what the engine **holds** before dispatch |
| `csv/sniffer_engine_1min_2024_04.csv` | runtime sniffer subscribed to `…-1-MINUTE-{BID,ASK}-EXTERNAL` | the 1-min bars `DataEngine` **dispatched** (same bars the `TimeBarAggregator` was fed) |
| `csv/sniffer_strategy_5min_2024_04.csv` | runtime sniffer subscribed to `…-5-MINUTE-{BID,ASK}-INTERNAL@1-MINUTE-EXTERNAL` | the 5-min bars the **aggregator emitted** to user-space `on_bar` |

All three share the same columns: `stream, price_type, bar_type, ts_event_ns, ts_init_ns, open, high, low, close, volume, ts_event, ts_init`.

The `cross-check raw == sniffer (engine side): missing=0 extra=0` line above is the proof: the engine held N 1-min bars in its merge-sorted buffer, dispatched all N at runtime, and the aggregator consumed exactly those N. The strategy's `on_bar` then received the aggregator's emitted 5-min bars — confirming the two-layer model end-to-end.

**A subtlety worth noting** about the 5-min row count: Nautilus' INTERNAL `TimeBarAggregator` is **clock-based**, not data-driven. It emits a 5-min bar at every 5-min wall-clock boundary that elapses during the run window, regardless of whether a 1-min input bar arrived in that window. For April 2024 (≈30 days × 288 5-min windows/day = 8,640 possible boundaries), expect ~8,500–8,600 5-min bars per side — *not* `count(1-min bars) / 5 = 31,350 / 5 ≈ 6,270`. The difference is the weekend / out-of-session gaps where there is no input 1-min data but the aggregator still ticks the clock. Compare against [aggregate_1min_to_5min.ipynb](aggregate_1min_to_5min.ipynb) where pandas-resample-with-dropna gives the data-driven count (~6,275).

This does **not** affect the strategy's `on_bar` logic: 5-min bars emitted in empty windows carry forward OHLC from the prior bar, so signals on dry windows are no-ops. But it does mean if you compare counts to a pandas resample, you'll see ~37% more rows from Nautilus' aggregator.

## Scenario D — Live paced visualizer (engine clock unaffected)

A real-time tick-by-tick view of what the engine dispatches (1-min bars) and what the strategy receives (5-min aggregator output), printed to stdout *during* `engine.run()`. Each print is paced by `time.sleep(...)` inside the sniffer's `on_bar` so you can watch the stream unfold one bar at a time.

**Why this is safe — the engine clock is event-driven, not wall-clock-driven.**

Nautilus' `BacktestEngine` uses a `TestClock` (a simulated/virtual clock). The run loop advances it only when it reads the next event's `ts_init` — see `_advance_time` at [venv/Lib/site-packages/nautilus_trader/backtest/engine.pyx:1684](venv/Lib/site-packages/nautilus_trader/backtest/engine.pyx#L1684). It never queries `time.time()`. So `time.sleep` inside `on_bar`:

| | Affected? |
| --- | --- |
| OS thread parks for N ms wall time | yes (that's the point) |
| Simulated clock advances | **no** |
| `bar.ts_event` / `bar.ts_init` values | **no** |
| Aggregator emission timestamps | **no** |
| Indicator updates, order fills, P&L | **no** |
| Total wall-clock duration of the cell | yes (longer) |

The backtest output is **identical bit-for-bit** to a zero-delay run. We're just slowing down our own perception of the dispatch.

Defaults: 1-hour window (2024-04-01 10:00–11:00 UTC, ~60 1-min bars), 50ms per 1-min bar → ~3 s wall time. Change `window_start`, `window_end`, `delay_ms_1min` to taste. To run the full month live, set `delay_ms_1min=0` (instant) or pick a wider window with a smaller delay.

In [ ]:
import time


class PacedSnifferConfig(StrategyConfig, frozen=True):
    bar_type: BarType
    label: str          # short prefix for the live feed line
    delay_ms: int = 0   # wall-clock pause after each on_bar (0 = no pause)


class PacedSnifferStrategy(Strategy):
    """Subscribe to one BarType; print every bar in real time, optionally pacing.

    The print + sleep happen INSIDE on_bar, which runs in the engine's
    synchronous dispatch loop. time.sleep() blocks the OS thread but does
    NOT advance the simulated TestClock — the engine clock is event-driven
    (see _advance_time at venv/Lib/site-packages/nautilus_trader/backtest/
    engine.pyx:1684), so backtest outputs are identical whether delay_ms
    is 0 or 100.
    """

    def on_start(self) -> None:
        self.subscribe_bars(self.config.bar_type)

    def on_bar(self, bar: Bar) -> None:
        ts = pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")
        print(
            f"[{self.config.label}] {ts}  "
            f"O={float(bar.open):.5f}  H={float(bar.high):.5f}  "
            f"L={float(bar.low):.5f}  C={float(bar.close):.5f}  "
            f"V={float(bar.volume):.0f}",
            flush=True,
        )
        if self.config.delay_ms > 0:
            time.sleep(self.config.delay_ms / 1000.0)

In [ ]:
def run_paced_visualizer(
    window_start: str = "2024-04-01 10:00",
    window_end:   str = "2024-04-01 11:00",
    delay_ms_1min: int = 50,
) -> None:
    """Run a small windowed backtest with live paced printing of both layers.

    Parameters
    ----------
    window_start, window_end
        ISO strings (UTC). Slice of df_1min_bid to feed the engine.
        Default is a 1-hour window on 2024-04-01 (~60 1-min bars).
    delay_ms_1min
        Wall-clock pause after each 1-min on_bar print.
        The 5-min sniffer prints with no extra delay.
        Total wall time ~= (n_1min_bars * delay_ms_1min) ms.

    Engine TestClock is NOT affected by the sleep — bar timestamps,
    aggregator emissions, and any backtest output are identical to a
    zero-delay run.
    """
    win_start = pd.Timestamp(window_start, tz="UTC")
    win_end   = pd.Timestamp(window_end,   tz="UTC")
    window_df = df_1min_bid.loc[(df_1min_bid.index >= win_start) & (df_1min_bid.index < win_end)]
    if window_df.empty:
        print(f"no data in window {window_start} .. {window_end}")
        return

    print(f"window     : {window_df.index.min()} .. {window_df.index.max()}  ({len(window_df):,} 1-min BID bars)")
    print(f"pacing     : {delay_ms_1min}ms per 1-min bar (wall-clock only)")
    print(f"engine clock: simulated, event-driven — unaffected by sleep")
    print("-" * 90)

    instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)
    bt_1min, bars_1min = build_bars(window_df, instrument, 1, "MINUTE", "BID")
    bt_5min = BarType.from_str(
        f"{instrument.id}-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL"
    )

    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("PACED-001"),
            logging=LoggingConfig(bypass_logging=True),
            risk_engine=RiskEngineConfig(bypass=True),
            run_analysis=False,
        )
    )
    engine.add_venue(
        venue=instrument.id.venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, USD)],
        base_currency=USD,
        default_leverage=Decimal(1),
    )
    engine.add_instrument(instrument)
    engine.add_data(bars_1min)

    # 1-min sniffer: paces the run by sleeping after each print.
    engine.add_strategy(PacedSnifferStrategy(PacedSnifferConfig(
        bar_type=bt_1min, label="engine 1-min ", delay_ms=delay_ms_1min,
    )))
    # 5-min sniffer: prints immediately on each aggregator emission, no extra delay.
    engine.add_strategy(PacedSnifferStrategy(PacedSnifferConfig(
        bar_type=bt_5min, label="strat  5-min ", delay_ms=10000,
    )))

    t0 = time.perf_counter()
    engine.run()
    elapsed = time.perf_counter() - t0
    print("-" * 90)
    sim_span = window_df.index.max() - window_df.index.min()
    print(f"engine.run() wall-clock: {elapsed:.2f}s  |  simulated window: {sim_span}")
    print(f"(simulated time is decoupled from wall time — sleep only paces visibility)")

    engine.dispose()


run_paced_visualizer()

window     : 2024-04-01 10:00:00+00:00 .. 2024-04-01 10:59:00+00:00  (60 1-min BID bars)
pacing     : 50ms per 1-min bar (wall-clock only)
engine clock: simulated, event-driven — unaffected by sleep
------------------------------------------------------------------------------------------
[engine 1-min ] 2024-04-01 10:00:00+00:00  O=1.07860  H=1.07865  L=1.07857  C=1.07863  V=142
[strat  5-min ] 2024-04-01 10:00:00+00:00  O=1.07860  H=1.07865  L=1.07857  C=1.07863  V=142
[engine 1-min ] 2024-04-01 10:01:00+00:00  O=1.07861  H=1.07866  L=1.07859  C=1.07863  V=104
[engine 1-min ] 2024-04-01 10:02:00+00:00  O=1.07862  H=1.07862  L=1.07857  C=1.07857  V=26
[engine 1-min ] 2024-04-01 10:03:00+00:00  O=1.07857  H=1.07857  L=1.07850  C=1.07851  V=43
[engine 1-min ] 2024-04-01 10:04:00+00:00  O=1.07851  H=1.07854  L=1.07847  C=1.07847  V=119
[engine 1-min ] 2024-04-01 10:05:00+00:00  O=1.07846  H=1.07846  L=1.07836  C=1.07836  V=71
[strat  5-min ] 2024-04-01 10:05:00+00:00  O=1.07861  H=1.0786

## Takeaway

Read the `latency_sec` columns above. In scenario A every value is a multiple of 300 s — the matching engine had nothing finer than 5-min bars to fill against. In scenario B the values are multiples of 60 s — even though the strategy still received only 5-min bars (verified by the fact that the EMA-cross signals are identical), the matching engine consumed every 1-min bar and filled the market orders at the very next 1-min boundary.

**So, precisely:** when 1-min EXTERNAL bars are loaded into the engine and the strategy subscribes to a `5-MIN-INTERNAL@1-MIN-EXTERNAL` bar type:

- The 1-min bars are **not** skipped by the engine. They drive `SimulatedExchange`, so stop/limit/bracket orders are matched at 1-min granularity.
- They **are** skipped by the strategy's `on_bar` handler. Anything that runs there — including the SL/TGT checks in [../core/managed_strategy.py](../core/managed_strategy.py) — sees only 5-min bars.

If you want SL/TGT checked at 1-min precision *and* signals on 5-min, subscribe to both via `EMACrossConfig.extra_bar_types` (already supported in [../strategies/ema_cross.py](../strategies/ema_cross.py#L21)). If you load only pre-aggregated 5-min EXTERNAL bars (scenario A), the entire engine — including the matching layer — is locked at 5-min granularity.